# RegimeShift: Macro-Aware Tactical Asset Allocation Engine

## Summer of Quant 2026 - Advanced Track

A complete pipeline for detecting market regimes using Hidden Markov Models and dynamically rebalancing a multi-asset portfolio.

---

### Project Phases
1. **Phase 1: Data Pipeline** ← You are here
2. **Phase 2: Feature Engineering**
3. **Phase 3: Regime Detection (HMM)**
4. **Phase 4: Portfolio Optimization**
5. **Phase 5: Backtesting & Benchmarking**

---

## Environment & Imports

In [ ]:
# Install missing packages (run once)
# !pip install yfinance hmmlearn cvxpy scikit-learn numpy pandas matplotlib scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import yfinance as yf
import scipy.stats as stats
from datetime import datetime, timedelta
import warnings

warnings.filterwarnings('ignore')

# Try to import ML libraries (used in Phase 3+)
try:
    from hmmlearn import hmm
    HMM_OK = True
except ImportError:
    HMM_OK = False
    print("⚠️ hmmlearn not installed - install with: pip install hmmlearn")

try:
    import cvxpy as cp
    CVXPY_OK = True
except ImportError:
    CVXPY_OK = False
    print("⚠️ cvxpy not installed - install with: pip install cvxpy")

# Plotting config
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

print("✅ Environment ready.")
print(f"   HMM support: {HMM_OK}")
print(f"   CVXPY support: {CVXPY_OK}")

---

# PHASE 1: DATA PIPELINE

**Goal**: Download clean, aligned daily prices for:
- **NSE (India equities)**: `^NSEI` (NIFTY 50 index)
- **Bonds**: `IEF` (US 7-10Y Treasuries, as proxy for Indian bond exposure)
- **Gold**: `GLD` (Gold ETF)
- **Volatility Index**: `^VIX` (CBOE VIX, market fear gauge)

Then convert to log-returns and align on a shared DatetimeIndex.

---

## Step 1: Download Multi-Asset Data

In [ ]:
# Configuration
TICKERS = ["^NSEI", "IEF", "GLD", "^VIX"]  # NSE, Bonds, Gold, VIX
START_DATE = "2015-01-01"
END_DATE = "2024-01-01"

print(f"Downloading {TICKERS} from {START_DATE} to {END_DATE}...")

# Download OHLCV data
raw_data = yf.download(
    TICKERS,
    start=START_DATE,
    end=END_DATE,
    progress=False,
    auto_adjust=True  # Auto-adjust for dividends/splits
)

print(f"✅ Downloaded {len(raw_data)} trading days")
print(f"\nRaw data shape: {raw_data.shape}")
print(f"Columns (MultiIndex): {raw_data.columns.names}")

## Step 2: Extract and Clean Close Prices

In [ ]:
# Extract just the Close prices
close_prices = raw_data["Close"].copy()
close_prices.columns.name = None  # Remove MultiIndex level name

# Rename VIX ticker for clarity
close_prices = close_prices.rename(columns={"^VIX": "VIX", "^NSEI": "NSE"})

print("Close prices (first 5 rows):")
print(close_prices.head())

print("\nMissing values per column:")
print(close_prices.isna().sum())

print(f"\nDate range: {close_prices.index[0].date()} to {close_prices.index[-1].date()}")
print(f"Total trading days: {len(close_prices)}")

## Step 3: Compute Log Returns for Asset Prices

**Note**: VIX is a level (not a price), so we don't return-transform it. We'll use the raw VIX level as a feature.

In [ ]:
# Asset prices (what we return-transform)
asset_prices = close_prices[["NSE", "IEF", "GLD"]].copy()

# Compute log returns: ln(P_t / P_t-1) = ln(P_t) - ln(P_t-1)
asset_returns = np.log(asset_prices).diff().dropna()

# VIX is kept as a level (not return-transformed)
vix_level = close_prices["VIX"].copy()

print("Asset log returns (first 5 rows):")
print(asset_returns.head())

print("\nBasic statistics (annualized):")
print(f"  NSE - Mean return: {asset_returns['NSE'].mean() * 252:.4f}, Vol: {asset_returns['NSE'].std() * np.sqrt(252):.4f}")
print(f"  IEF - Mean return: {asset_returns['IEF'].mean() * 252:.4f}, Vol: {asset_returns['IEF'].std() * np.sqrt(252):.4f}")
print(f"  GLD - Mean return: {asset_returns['GLD'].mean() * 252:.4f}, Vol: {asset_returns['GLD'].std() * np.sqrt(252):.4f}")

print("\nCorrelation matrix:")
print(asset_returns.corr().round(3))

## Step 4: Align All Series on Shared DatetimeIndex

In [ ]:
# Inner join: keep only dates where ALL assets have data
master_df = asset_returns.join(vix_level, how="inner").dropna()

print(f"Aligned dataset shape: {master_df.shape}")
print(f"Date range: {master_df.index[0].date()} to {master_df.index[-1].date()}")

print("\nFirst 5 rows:")
print(master_df.head())

print("\nLast 5 rows:")
print(master_df.tail())

print("\nData types:")
print(master_df.dtypes)

print(f"\n✅ Phase 1 complete: {len(master_df)} days of clean, aligned data")

## Step 5: Visualize Raw Price Movements

Sanity check: do the asset classes behave differently in calm vs. stressed periods?

In [ ]:
# Compute cumulative returns from the aligned data
cumulative_returns = np.exp(master_df[["NSE", "IEF", "GLD"]].cumsum())

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(cumulative_returns.index, cumulative_returns["NSE"], label="NSE (Equities)", linewidth=1.5)
ax.plot(cumulative_returns.index, cumulative_returns["IEF"], label="IEF (Bonds)", linewidth=1.5)
ax.plot(cumulative_returns.index, cumulative_returns["GLD"], label="GLD (Gold)", linewidth=1.5)

# Shade known crisis periods
crisis_periods = [
    ("2015-08-24", "2015-08-31", "Aug 2015\nChina"),
    ("2018-12-01", "2018-12-31", "Dec 2018\nVol Spike"),
    ("2020-02-15", "2020-03-31", "COVID-19\nCrash"),
    ("2022-02-01", "2022-10-31", "2022\nInflation"),
]

for start, end, label in crisis_periods:
    try:
        ax.axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.15, color="red")
    except:
        pass

ax.set_title("Cumulative Returns: NSE vs Bonds vs Gold (Known Crisis Periods Shaded)")
ax.set_xlabel("Date")
ax.set_ylabel("Growth of $1")
ax.legend(loc="upper left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("💡 Observations:")
print("  • NSE (equities) is most volatile")
print("  • Bonds (IEF) provide some cushioning but correlate with equities in crashes")
print("  • Gold shows relative resilience in crisis periods (diversification benefit)")
print("  • These patterns will be captured by the HMM in Phase 3")

## Step 6: VIX Behavior Check

VIX should spike during crisis periods. This validates our data quality and gives us intuition for regime detection.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(master_df.index, master_df["VIX"], color="darkred", linewidth=0.8, label="VIX")
ax.axhline(20, color="orange", linestyle="--", linewidth=1, alpha=0.7, label="Elevated (20)")
ax.axhline(30, color="red", linestyle="--", linewidth=1, alpha=0.7, label="Extreme (30)")

# Mark known crisis periods
for start, end, label in crisis_periods:
    try:
        ax.axvspan(pd.to_datetime(start), pd.to_datetime(end), alpha=0.1, color="red")
    except:
        pass

ax.set_title("VIX (Volatility Index): Should spike during crisis periods")
ax.set_xlabel("Date")
ax.set_ylabel("VIX Level")
ax.legend(loc="upper right")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print(f"VIX stats:")
print(f"  Mean: {master_df['VIX'].mean():.2f}")
print(f"  Median: {master_df['VIX'].median():.2f}")
print(f"  Min: {master_df['VIX'].min():.2f}")
print(f"  Max: {master_df['VIX'].max():.2f}")
print(f"\n✅ Data looks good! VIX spikes align with known crisis periods.")

## Step 7: Save Aligned Dataset for Phase 2

This is our single source of truth for all downstream phases.

In [ ]:
# Save to CSV for reproducibility
master_df.to_csv("data/master_returns.csv")
print(f"✅ Saved aligned dataset to data/master_returns.csv")

# Also save the original close prices (useful for portfolio backtesting later)
asset_prices_aligned = asset_prices.loc[master_df.index]
asset_prices_aligned.to_csv("data/asset_prices.csv")
print(f"✅ Saved aligned prices to data/asset_prices.csv")

print(f"\n📊 Summary:")
print(f"  • Total observations: {len(master_df)}")
print(f"  • Date range: {master_df.index[0].date()} to {master_df.index[-1].date()}")
print(f"  • Features: {list(master_df.columns)}")
print(f"  • All data is CLEAN (no NaNs, properly aligned)")
print(f"\n✅ PHASE 1 COMPLETE: Data Pipeline Ready")

---

## Next: Phase 2 - Feature Engineering

In the next phase, we'll engineer:
- **Momentum features** (rolling returns at multiple horizons: 5d, 21d, 63d, 126d)
- **Volatility features** (rolling std of returns, annualized)
- **Safe z-scoring** (using expanding windows to avoid lookahead bias)

These features will feed into the HMM in Phase 3.

---